# Model Building and Training

Pipeline: Logistic Regression baseline → XGBoost ensemble → Cross-validation → Comparison.

**Primary metrics:** AUC-PR, F1-Score (accuracy is misleading on imbalanced data)

In [ ]:
import sys
sys.path.insert(0, '..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from src.modeling import (
    build_logistic_regression, build_xgboost,
    train_and_evaluate, cross_val_evaluate, save_model,
    plot_confusion_matrix, plot_precision_recall_curve, compare_models
)
from src.resampling import get_class_distribution, apply_smote
plt.rcParams['figure.dpi'] = 120
print('Setup complete')

## Dataset A: E-Commerce Fraud

In [ ]:
X_train = pd.read_csv('../data/processed/fraud_X_train.csv')
y_train = pd.read_csv('../data/processed/fraud_y_train.csv').squeeze()
X_test  = pd.read_csv('../data/processed/fraud_X_test.csv')
y_test  = pd.read_csv('../data/processed/fraud_y_test.csv').squeeze()
print(f'Train: {X_train.shape} | Test: {X_test.shape}')

In [ ]:
# Logistic Regression baseline
lr_fraud = build_logistic_regression()
lr_metrics = train_and_evaluate(lr_fraud, X_train, y_train, X_test, y_test, 'LogisticRegression (Fraud)')
save_model(lr_fraud, 'lr_fraud')

In [ ]:
# XGBoost
spw = (y_train == 0).sum() / (y_train == 1).sum()
xgb_fraud = build_xgboost(scale_pos_weight=spw)
xgb_metrics = train_and_evaluate(xgb_fraud, X_train, y_train, X_test, y_test, 'XGBoost (Fraud)')
save_model(xgb_fraud, 'xgb_fraud')

In [ ]:
# Confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
plot_confusion_matrix(lr_metrics['confusion_matrix'], 'LR — Fraud Data', axes[0])
plot_confusion_matrix(xgb_metrics['confusion_matrix'], 'XGBoost — Fraud Data', axes[1])
plt.tight_layout(); plt.show()

In [ ]:
# PR Curves
fig, ax = plt.subplots(figsize=(7, 5))
plot_precision_recall_curve(y_test, lr_fraud.predict_proba(X_test)[:,1], 'Logistic Regression', ax)
plot_precision_recall_curve(y_test, xgb_fraud.predict_proba(X_test)[:,1], 'XGBoost', ax)
plt.title('Precision-Recall Curves — E-Commerce Fraud')
plt.legend(); plt.show()

In [ ]:
# 5-Fold CV
X_all = pd.concat([X_train, X_test])
y_all = pd.concat([y_train, y_test])
print('=== CV: Logistic Regression ===')
cross_val_evaluate(build_logistic_regression(), X_all, y_all, k=5)
print('\n=== CV: XGBoost ===')
cross_val_evaluate(build_xgboost(scale_pos_weight=spw), X_all, y_all, k=5)

## Dataset B: Credit Card Transactions

In [ ]:
cc_df = pd.read_csv('../data/processed/creditcard_cleaned.csv')
X_cc = cc_df.drop(columns=['Class'])
y_cc = cc_df['Class']
scaler_cc = StandardScaler()
X_cc[['Time', 'Amount']] = scaler_cc.fit_transform(X_cc[['Time', 'Amount']])
X_tr, X_te, y_tr, y_te = train_test_split(X_cc, y_cc, test_size=0.2, stratify=y_cc, random_state=42)
X_tr_res, y_tr_res = apply_smote(X_tr, y_tr)
print(f'Resampled train shape: {X_tr_res.shape}')

In [ ]:
lr_cc = build_logistic_regression()
lr_cc_m = train_and_evaluate(lr_cc, X_tr_res, y_tr_res, X_te, y_te, 'LR (CreditCard)')
save_model(lr_cc, 'lr_creditcard')

spw_cc = (y_tr == 0).sum() / (y_tr == 1).sum()
xgb_cc = build_xgboost(scale_pos_weight=spw_cc)
xgb_cc_m = train_and_evaluate(xgb_cc, X_tr_res, y_tr_res, X_te, y_te, 'XGBoost (CreditCard)')
save_model(xgb_cc, 'xgb_creditcard')

In [ ]:
print('\n=== E-Commerce Comparison ===')
compare_models([lr_metrics, xgb_metrics])
print('\n=== Credit Card Comparison ===')
compare_models([lr_cc_m, xgb_cc_m])

## Model Selection

**Selected: XGBoost** for both datasets.
- Higher AUC-PR (primary metric) on both datasets
- Handles non-linear interactions (velocity × time × geography)
- Built-in feature importance for pre-SHAP inspection

**Next step:** `shap-explainability.ipynb`